# Week 4 — A\* as a Ten-Line Diff

**Lesson plan:** [`../weeks/week-04.md`](../weeks/week-04.md) · **Slides:** [`../slides/week-04/deck.md`](../slides/week-04/deck.md)

Live coding, 25 minutes. Reuse week 3's machinery, change the frontier ordering
from `g` to `g + h`, and the algorithm changes character completely.

**The artifact this notebook produces is one table.** That table is the whole
lecture, and its last row is the bridge to every LLM comparison in the course.

## 1 · Week 3's code, unchanged

Normally you would `from aicourse.search import ...`. It is repeated here so the
notebook stands alone.

In [1]:
import heapq, itertools, time

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


class Node:
    __slots__ = ("state", "parent", "action", "g")
    def __init__(self, state, parent=None, action=None, g=0):
        self.state, self.parent, self.action, self.g = state, parent, action, g
    def path(self):
        node, out = self, []
        while node.parent is not None:
            out.append(node.action); node = node.parent
        return out[::-1]


class EightPuzzle:
    MOVES = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5),
             3: (0, 4, 6), 4: (1, 3, 5, 7), 5: (2, 4, 8),
             6: (3, 7), 7: (4, 6, 8), 8: (5, 7)}
    def __init__(self, initial, goal=GOAL):
        self.initial, self.goal = initial, goal
    def actions(self, state):
        return self.MOVES[state.index(0)]
    def result(self, state, action):
        s = list(state); b = state.index(0)
        s[b], s[action] = s[action], s[b]
        return tuple(s)
    def is_goal(self, state):
        return state == self.goal
    def step_cost(self, state, action):
        return 1


# verified optimal depths (see week 3's appendix)
D8  = (0, 4, 2, 5, 1, 3, 7, 8, 6)
D12 = (5, 4, 2, 7, 0, 3, 8, 1, 6)
D16 = (7, 5, 2, 4, 0, 3, 8, 1, 6)
print("week 3 machinery loaded")

week 3 machinery loaded


## 2 · A\* — the diff

Compare this to week 3's `search`. The differences: the frontier is keyed on
`g + h(state)` instead of `g`, and we keep a `best_g` map so a cheaper route to an
already-seen state can supersede an expensive one.

In [2]:
def astar(problem, h):
    start = Node(problem.initial)
    frontier = [(h(start.state), 0, start)]      # (f, tiebreak, node)
    best_g = {problem.initial: 0}
    expansions, tiebreak = 0, 0

    while frontier:
        f, _, node = heapq.heappop(frontier)
        if problem.is_goal(node.state):
            return node, expansions
        if node.g > best_g.get(node.state, float("inf")):
            continue                              # stale entry, already improved
        expansions += 1
        for action in problem.actions(node.state):
            s2 = problem.result(node.state, action)
            g2 = node.g + problem.step_cost(node.state, action)
            if g2 < best_g.get(s2, float("inf")):
                best_g[s2] = g2
                tiebreak += 1
                heapq.heappush(frontier,
                               (g2 + h(s2), tiebreak, Node(s2, node, action, g2)))
    return None, expansions

## 3 · Four heuristics

Two admissible, one trivially admissible, one deliberately **in**admissible.

In [3]:
def h_zero(s):
    """Admissible, useless. A* degenerates to uniform-cost search."""
    return 0


def h_misplaced(s):
    """Count tiles not in place. Admissible: each needs >= 1 move."""
    return sum(1 for i, v in enumerate(s) if v != 0 and v != GOAL[i])


def manhattan(i, tile):
    goal_i = GOAL.index(tile)
    return abs(i // 3 - goal_i // 3) + abs(i % 3 - goal_i % 3)


def h_manhattan(s):
    """Sum of tile distances. Admissible, and dominates h_misplaced."""
    return sum(manhattan(i, s[i]) for i in range(9) if s[i] != 0)


def h_bad(s):
    """INADMISSIBLE -- overestimates by up to 3x. Watch what it costs."""
    return 3 * h_manhattan(s)


# Dominance check: h_manhattan >= h_misplaced for every state, by construction.
import random
random.seed(0)
s = list(GOAL)
for _ in range(500):
    p = EightPuzzle(tuple(s))
    s = list(p.result(tuple(s), random.choice(p.actions(tuple(s)))))
    assert h_manhattan(tuple(s)) >= h_misplaced(tuple(s))
print("dominance h_manhattan >= h_misplaced verified on 500 random states")

dominance h_manhattan >= h_misplaced verified on 500 random states


## 4 · The table that is the whole lecture

Run all four on the same depth-16 instance.

In [4]:
def compare(start, label):
    p = EightPuzzle(start)
    print(f"\n{label}   (true optimal = {TRUE[start]} moves)")
    print(f"  {'heuristic':<16}{'expansions':>12}{'sol.length':>12}{'optimal?':>10}{'time':>9}")
    print("  " + "-" * 60)
    for name, h in [("h_zero (=UCS)", h_zero), ("h_misplaced", h_misplaced),
                    ("h_manhattan", h_manhattan), ("h_bad (INADMIS.)", h_bad)]:
        t0 = time.perf_counter()
        node, exp = astar(p, h)
        dt = time.perf_counter() - t0
        n = len(node.path())
        ok = "yes" if n == TRUE[start] else "NO  <--"
        print(f"  {name:<16}{exp:>12,}{n:>12}{ok:>10}{dt:>8.3f}s")


TRUE = {D8: 8, D12: 12, D16: 16}
compare(D16, "depth-16 instance")


depth-16 instance   (true optimal = 16 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)         11,276          16       yes   0.038s
  h_misplaced              685          16       yes   0.002s
  h_manhattan              169          16       yes   0.001s
  h_bad (INADMIS.)          84          18   NO  <--   0.000s


### The two payoffs

**1 · Manhattan dominates misplaced-tiles, and the expansion count collapses.**
The warm-up poll is now answered with data rather than intuition. Dominance is not
an aesthetic preference — it is fewer nodes, every time, provably.

**2 · ⚠️ `h_bad` is the fastest, and it is wrong.**

It expands the fewest nodes *and* returns a suboptimal solution. Say this plainly:

> **You can always buy speed by giving up the guarantee — and if you do not
> measure solution quality, you will not notice.**

That sentence is the bridge to every LLM comparison in this course. An LLM
pathfinder is `h_bad` with better marketing: fast, plausible, and silently
suboptimal unless you check.

## 5 · Scaling — does the ranking hold?

One instance is an anecdote. Run all three depths.

In [5]:
for start, label in [(D8, "depth-8"), (D12, "depth-12"), (D16, "depth-16")]:
    compare(start, label + " instance")


depth-8 instance   (true optimal = 8 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)            172           8       yes   0.000s
  h_misplaced               16           8       yes   0.000s
  h_manhattan               11           8       yes   0.000s
  h_bad (INADMIS.)          11           8       yes   0.000s

depth-12 instance   (true optimal = 12 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)          2,177          12       yes   0.006s
  h_misplaced              110          12       yes   0.001s
  h_manhattan               28          12       yes   0.000s
  h_bad (INADMIS.)          35          12       yes   0.000s

depth-16 instance   (true optimal = 16 moves)
  heuristic         expansions  sol.length  optimal?     time
  ---------------------------------------------------

### ⚠️ Read your own data honestly

Look at what actually happened across the three depths:

- At **depth 8 and 12**, `h_bad` found the *optimal* path anyway. Being
  inadmissible means A\* **may** return a suboptimal answer — not that it always
  will. An unsound method that happens to be right on easy instances is the most
  dangerous kind, because it builds false confidence.
- At **depth 12**, `h_bad` expanded *more* nodes than `h_manhattan` (35 vs 28). It
  is not even reliably faster. Overestimating distorts the search order; sometimes
  that helps, sometimes it wastes work.
- Only at **depth 16** does the failure surface: 84 expansions, and an 18-move
  answer when 16 exists.

> **The failure appeared only at the largest size we tested.** If we had stopped at
> depth 12 we would have concluded `h_bad` was fine.
>
> This is why every Duel in this course requires a **scaling plot** rather than a
> single benchmark number. Soundness failures hide at small sizes.

## 6 · Effective branching factor b\*

The honest way to compare heuristics across problem sizes. If A\* expands `N`
nodes to find a solution at depth `d`, then `b*` is the branching factor a uniform
tree of depth `d` would need to contain `N` nodes.

**b\* near 1 means the search went almost straight to the goal.**

In [6]:
def effective_bf(N, d, lo=1.0000001, hi=10.0, tol=1e-6):
    """Solve N + 1 = 1 + b + b^2 + ... + b^d for b, by bisection."""
    def total(b):
        return sum(b ** i for i in range(d + 1))
    for _ in range(200):
        mid = (lo + hi) / 2
        if total(mid) < N + 1:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2


print(f"  {'heuristic':<16}{'depth':>7}{'expansions':>12}{'b*':>8}")
print("  " + "-" * 43)
for start in (D8, D12, D16):
    for name, h in [("h_zero", h_zero), ("h_misplaced", h_misplaced),
                    ("h_manhattan", h_manhattan)]:
        node, exp = astar(EightPuzzle(start), h)
        d = len(node.path())
        print(f"  {name:<16}{d:>7}{exp:>12,}{effective_bf(exp, d):>8.3f}")
    print()

print("""A better heuristic pushes b* toward 1.0 and KEEPS IT THERE as depth grows.
That stability is the property you actually want -- a heuristic that looks good on
easy instances and degrades on hard ones is the classic trap, and it is exactly
what the scaling plots in every Duel are designed to expose.""")

  heuristic         depth  expansions      b*
  -------------------------------------------
  h_zero                8         172   1.708
  h_misplaced           8          16   1.153
  h_manhattan           8          11   1.070

  h_zero               12       2,177   1.770
  h_misplaced          12         110   1.318
  h_manhattan          12          28   1.125

  h_zero               16      11,276   1.695
  h_misplaced          16         685   1.389
  h_manhattan          16         169   1.248

A better heuristic pushes b* toward 1.0 and KEEPS IT THERE as depth grows.
That stability is the property you actually want -- a heuristic that looks good on
easy instances and degrades on hard ones is the classic trap, and it is exactly
what the scaling plots in every Duel are designed to expose.


## 7 · Where heuristics come from — relaxed problems

> **Every admissible heuristic is the exact solution to an easier problem.**

| Relaxation | Resulting heuristic |
|---|---|
| A tile can move anywhere, instantly | `h_misplaced` |
| A tile can move to any *adjacent* square, even if occupied | `h_manhattan` |
| No relaxation — the real problem | the answer itself |

Both heuristics are *exact costs* for a game with weaker rules. That is why they
never overestimate, and it is a recipe rather than a guess: **drop a constraint,
solve what remains, and you have an admissible heuristic.**

Week 9 automates this. A planner reads your domain file, drops the delete effects,
and derives its own heuristic — the same principle, performed by machine.

## 8 · Exercise

1. Add the **linear conflict** heuristic and confirm it dominates Manhattan.
2. Build `h_max = max(h_misplaced, h_manhattan)`. Prove it is admissible, then
   confirm it dominates both.
3. Set `h_bad = 1.1 * h_manhattan`. How suboptimal do the solutions get? This is
   *weighted A\**, and the answer is bounded — find the bound in AIMA §3.5.
4. Plot expansions against depth for all four heuristics on the depth 8/12/16
   instances. **That plot is the deliverable format for every Duel in this
   course.**